# Minimal exporter for the six named-regime animations

This notebook only loads the six previously saved `.pkl.gz` simulation bundles,
creates one dual-view animation at a time, saves it as MP4, and releases it from memory.

It does not rerun the opinion-dynamics simulations and does not display animations inline.

In [1]:
from pathlib import Path

REGIME_DATA_DIR = Path("named_regime_simulations")
OUTPUT_DIR = Path("named_regime_mp4s")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FRAME_STEP = 4
FPS = 8
DPI = 110
INTERVAL_MS = 125
NODE_SIZE_MIN = 25
NODE_SIZE_MAX = 110
EDGE_HIGHLIGHT_FRAMES = 1

In [2]:
import gc
import gzip
import pickle
import re

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

try:
    import imageio_ffmpeg
    plt.rcParams["animation.ffmpeg_path"] = imageio_ffmpeg.get_ffmpeg_exe()
except ImportError:
    pass

In [3]:
REGIME_NAMES = [
    "Diverse weak-personalization environment",
    "Moderately personalized environment",
    "Algorithmic filter bubble",
    "Popularity-dominated mainstream",
    "Socially dominated echo chamber",
    "Cross-cutting high-openness environment"
]

def safe_filename(name):
    slug = re.sub(r"[^A-Za-z0-9]+", "_", name.strip()).strip("_").lower()
    return slug or "regime"

def regime_file_path(regime_name):
    return REGIME_DATA_DIR / f"{safe_filename(regime_name)}.pkl.gz"

def load_simulation_bundle(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Missing saved simulation: {path}\n"
            "Copy the named_regime_simulations folder next to this notebook."
        )
    with gzip.open(path, "rb") as file:
        return pickle.load(file)

In [4]:
def canonical_edge(edge):
    u, v = edge
    return (u, v) if u < v else (v, u)

def active_graph_snapshot_time(graph_history, time_step):
    valid_times = [t for t in graph_history if t <= time_step]
    if not valid_times:
        raise ValueError(f"No graph snapshot exists at or before t={time_step}.")
    return max(valid_times)

def previous_graph_snapshot_time(graph_history, time_step):
    current_time = active_graph_snapshot_time(graph_history, time_step)
    earlier = [t for t in graph_history if t < current_time]
    return max(earlier) if earlier else None

def edge_changes_at_time(graph_history, time_step):
    current_time = active_graph_snapshot_time(graph_history, time_step)
    current_graph = graph_history[current_time]
    previous_time = previous_graph_snapshot_time(graph_history, time_step)
    current_edges = {canonical_edge(e) for e in current_graph.edges()}

    if previous_time is None:
        return current_graph, None, current_edges, set(), set()

    previous_graph = graph_history[previous_time]
    previous_edges = {canonical_edge(e) for e in previous_graph.edges()}

    return (
        current_graph,
        previous_graph,
        current_edges & previous_edges,
        current_edges - previous_edges,
        previous_edges - current_edges
    )

def animation_frames(history_length, frame_step):
    frames = list(range(0, history_length, frame_step))
    if frames[-1] != history_length - 1:
        frames.append(history_length - 1)
    return frames

In [5]:
def mean_pairwise_distance(opinions):
    opinions = np.asarray(opinions)
    if len(opinions) < 2:
        return 0.0
    distances = np.abs(opinions[:, None] - opinions[None, :])
    return float(np.mean(distances[np.triu_indices(len(opinions), k=1)]))

def live_animation_metrics(history, graph, time_step):
    opinions = history[time_step]
    initial = history[0]
    pairwise = mean_pairwise_distance(opinions)
    extremity = np.mean(np.abs(opinions))
    nonneutral_initial = np.abs(initial) > 0.02
    opposite_side = nonneutral_initial & (initial * opinions < 0)

    if graph.number_of_edges() > 0:
        edge_distance = np.mean([
            abs(opinions[i] - opinions[j]) for i, j in graph.edges()
        ])
    else:
        edge_distance = np.nan

    return {
        "polarization": pairwise * extremity,
        "opposite_side": int(opposite_side.sum()),
        "edge_distance": edge_distance
    }

In [6]:
def make_dual_view_animation(
    bundle,
    frame_step=FRAME_STEP,
    interval=INTERVAL_MS,
    node_size_min=NODE_SIZE_MIN,
    node_size_max=NODE_SIZE_MAX,
    edge_highlight_frames=EDGE_HIGHLIGHT_FRAMES
):
    regime_name = bundle["regime_name"]
    history = np.asarray(bundle["history"])
    diagnostics = bundle["diagnostics"]
    spring_positions = bundle["spring_layout"]
    y_jitter = np.asarray(bundle["opinion_y_jitter"])

    graph_history = diagnostics["graph_history"]
    stubbornness_history = np.asarray(diagnostics["stubbornness_history"])
    frames = animation_frames(history.shape[0], frame_step)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.8))

    norm = Normalize(vmin=-1, vmax=1)
    cmap = plt.get_cmap("coolwarm")
    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    colorbar = fig.colorbar(sm, ax=axes, shrink=0.78, pad=0.02)
    colorbar.set_label("Opinion")

    def update(time_step):
        for ax in axes:
            ax.clear()

        opinions = history[time_step]
        stubbornness = stubbornness_history[time_step]
        sizes = node_size_min + stubbornness * (node_size_max - node_size_min)

        current_graph, previous_graph, persistent_edges, added_edges, removed_edges = (
            edge_changes_at_time(graph_history, time_step)
        )

        snapshot_time = active_graph_snapshot_time(graph_history, time_step)
        show_changes = (
            time_step - snapshot_time <= frame_step * edge_highlight_frames
        )

        if not show_changes:
            persistent_edges = {canonical_edge(e) for e in current_graph.edges()}
            added_edges = set()
            removed_edges = set()
            previous_graph = None

        opinion_positions = {
            node: (opinions[node], y_jitter[node])
            for node in current_graph.nodes()
        }

        current_nodes = list(current_graph.nodes())
        node_colors = [opinions[node] for node in current_nodes]
        node_sizes = [sizes[node] for node in current_nodes]

        nx.draw_networkx_edges(
            current_graph, opinion_positions,
            edgelist=list(persistent_edges),
            ax=axes[0], alpha=0.10, width=0.5, edge_color="gray"
        )
        if added_edges:
            nx.draw_networkx_edges(
                current_graph, opinion_positions,
                edgelist=list(added_edges),
                ax=axes[0], alpha=0.9, width=1.8, edge_color="green"
            )
        if previous_graph is not None and removed_edges:
            nx.draw_networkx_edges(
                previous_graph, opinion_positions,
                edgelist=list(removed_edges),
                ax=axes[0], alpha=0.65, width=1.2,
                edge_color="red", style="dashed"
            )

        nx.draw_networkx_nodes(
            current_graph, opinion_positions, ax=axes[0],
            nodelist=current_nodes, node_color=node_colors,
            cmap=cmap, vmin=-1, vmax=1, node_size=node_sizes,
            edgecolors="black", linewidths=0.15
        )

        axes[0].axvline(0, linestyle="--", linewidth=0.8, alpha=0.45)
        axes[0].set_xlim(-1.05, 1.05)
        axes[0].set_ylim(-0.32, 0.32)
        axes[0].set_yticks([])
        axes[0].set_xlabel("Opinion")
        axes[0].set_title("Opinion-space layout")

        nx.draw_networkx_edges(
            current_graph, spring_positions,
            edgelist=list(persistent_edges),
            ax=axes[1], alpha=0.10, width=0.5, edge_color="gray"
        )
        if added_edges:
            nx.draw_networkx_edges(
                current_graph, spring_positions,
                edgelist=list(added_edges),
                ax=axes[1], alpha=0.9, width=1.8, edge_color="green"
            )
        if previous_graph is not None and removed_edges:
            nx.draw_networkx_edges(
                previous_graph, spring_positions,
                edgelist=list(removed_edges),
                ax=axes[1], alpha=0.65, width=1.2,
                edge_color="red", style="dashed"
            )

        nx.draw_networkx_nodes(
            current_graph, spring_positions, ax=axes[1],
            nodelist=current_nodes, node_color=node_colors,
            cmap=cmap, vmin=-1, vmax=1, node_size=node_sizes,
            edgecolors="black", linewidths=0.15
        )

        axes[1].set_title("Fixed network layout")
        axes[1].set_axis_off()

        metrics = live_animation_metrics(history, current_graph, time_step)

        fig.suptitle(
            f"{regime_name} | Time = {time_step}\n"
            f"Polarization = {metrics['polarization']:.3f} | "
            f"Opposite initial side = {metrics['opposite_side']} | "
            f"Mean edge distance = {metrics['edge_distance']:.3f}"
        )

        axes[0].text(
            0.02, 0.03,
            "Color = opinion\n"
            "Size = stubbornness\n"
            "Green = added edge\n"
            "Red dashed = removed edge",
            transform=axes[0].transAxes,
            verticalalignment="bottom",
            fontsize=8,
            bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.80}
        )

    animation = FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=interval,
        repeat=False,
        cache_frame_data=False
    )

    return animation, fig

## Export all six MP4 files

The loop below processes one regime at a time and frees memory before loading the next.

In [7]:
missing_files = [
    regime_file_path(name)
    for name in REGIME_NAMES
    if not regime_file_path(name).exists()
]

if missing_files:
    missing_text = "\n".join(str(path) for path in missing_files)
    raise FileNotFoundError(
        "The following saved simulation files are missing:\n"
        f"{missing_text}\n\n"
        "Copy the named_regime_simulations folder beside this notebook."
    )

writer = FFMpegWriter(
    fps=FPS,
    metadata={
        "title": "Model 4C named-regime animation",
        "artist": "Virtual Earth project"
    },
    bitrate=1800
)

exported_files = []

for index, regime_name in enumerate(REGIME_NAMES, start=1):
    input_path = regime_file_path(regime_name)
    output_path = OUTPUT_DIR / f"{safe_filename(regime_name)}_dual_view.mp4"

    print(f"[{index}/{len(REGIME_NAMES)}] Loading: {regime_name}")

    bundle = load_simulation_bundle(input_path)
    animation, figure = make_dual_view_animation(bundle)

    print(f"    Saving: {output_path}")

    animation.save(
        output_path,
        writer=writer,
        dpi=DPI
    )

    exported_files.append(output_path)

    plt.close(figure)
    del animation, figure, bundle
    gc.collect()

    print("    Done.")

print("\nFinished exporting:")
for path in exported_files:
    print(path)

FileNotFoundError: The following saved simulation files are missing:
named_regime_simulations/diverse_weak_personalization_environment.pkl.gz
named_regime_simulations/moderately_personalized_environment.pkl.gz
named_regime_simulations/algorithmic_filter_bubble.pkl.gz
named_regime_simulations/popularity_dominated_mainstream.pkl.gz
named_regime_simulations/socially_dominated_echo_chamber.pkl.gz
named_regime_simulations/cross_cutting_high_openness_environment.pkl.gz

Copy the named_regime_simulations folder beside this notebook.

### If memory is still too high

Change the configuration near the top to:

```python
FRAME_STEP = 8
DPI = 85
NODE_SIZE_MIN = 15
NODE_SIZE_MAX = 75
```

This cuts the number of rendered frames and lowers per-frame memory use.